In [ ]:
import os
os.chdir("../")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
#Extract Data From the PDF File
def load_pdf_file(data):
    loader= DirectoryLoader(data,
                            glob="*.pdf",
                            loader_cls=PyPDFLoader)

    documents=loader.load()

    return documents

In [ ]:
extracted_data=load_pdf_file(data='Data/')

In [ ]:
#Split the Data into Text Chunks
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [ ]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
# Install via: pip install -U langchain-huggingface
from langchain_community.embeddings import HuggingFaceEmbeddings

def download_hugging_face_embeddings():
    # Use a real model path from Hugging Face
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. First, call the function to get the object
embeddings = download_hugging_face_embeddings()

# 2. Now that 'embeddings' exists, you can use it
query_result = embeddings.embed_query("Hello world")
print("Length:", len(query_result))

In [ ]:
import sys
!{sys.executable} -m pip install -U langchain-google-genai

In [ ]:
import langchain_google_genai
print("Installation verified and library found.")

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")


In [ ]:
import os
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key="pcsk_4JYGMS_22s7AY5S5PYjqVsQXufyB2xTKGfABwuPivVwsTGm1nty5cj9cFVWHoJJTq237XU")
index_name = "medical-doctorai"

# Check existing indexes using list_indexes().names()
existing_indexes = [index.name for index in pc.list_indexes()]

if index_name not in existing_indexes:
    print(f"Creating new index: {index_name}")
    pc.create_index(
        name=index_name,
        dimension=384, 
        metric="cosine", 
        spec=ServerlessSpec(
            cloud="aws", 
            region="us-east-1"
        )
    )
else:
    print(f"Index '{index_name}' already exists. Connecting to existing index.")

# Now you can safely target the index for upserts/queries
index = pc.Index(index_name)

In [ ]:
import os
if PINECONE_API_KEY:
    os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
    os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
    os.environ["GENAI_API_KEY"] = GEMINI_API_KEY


In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    index_name=index_name,
)

In [ ]:
# Load Existing index 

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [ ]:
docsearch

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
retrieved_search = retriever.invoke("What is acne?")

In [ ]:
retrieved_search

In [39]:
from langchain_google_genai import ChatGoogleGenerativeAI

# New Gemini code
# gemini-2.0 is not available for this API version; use a currently supported Gemini model.
llm = ChatGoogleGenerativeAI(
    api_key=os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY") or os.environ.get("GENAI_API_KEY"),
    model="gemini-2.0",
    temperature=0
)


In [40]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """You are an assistant that helps medical professionals understand clinical trials. \
You will be provided with relevant information from a medical book and your task is to answer questions based on that information.

Use the following context to answer the question. If the context does not contain the answer, respond with 'I don't know.'

{context}"""

# create_retrieval_chain passes the user question as "input", not "question"
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [41]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)

# Re-run this cell if you previously built a custom rag_chain with RunnablePassthrough.
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [ ]:
response = rag_chain.invoke({"input": "What is acne?"})
print(response["answer"])

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-2.0 is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

: 

In [ ]:
response = rag_chain.invoke({"input": "What is stats?"})
print(response["answer"])